# Behind the scenes: Day One
### -> How the data was downloaded

In [ ]:
# directory setup
from pathlib import Path

ROOT = Path("data")
RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"

for d in [RAW, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

### Berkeley Earth Surface Temperature (BEST) Dataset

In [ ]:
import pandas as pd
import requests

url = "https://berkeleyearth.lbl.gov/auto/Global/Complete_TAVG_complete.txt"

out_path = RAW / "berkeley_earth_global_temp.txt"

r = requests.get(url)
out_path.write_text(r.text)

In [ ]:
import pandas as pd

df = pd.read_csv(
    RAW / "berkeley_earth_global_temp.txt",
    skiprows=59,   # header offset in BEST files
    sep=r"\s+"
)

df.head()

In [ ]:
df_year = df.groupby("Year").mean(numeric_only=True).reset_index()

df_year.to_csv(PROCESSED / "best_global_yearly.csv", index=False)

### NASA GISTEMP Surface Temperature Dataset

In [ ]:
gistemp_url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"

gistemp_raw = RAW / "gistemp_global.csv"

import requests
gistemp_raw.write_text(requests.get(gistemp_url).text)

In [ ]:
import pandas as pd

gistemp = pd.read_csv(gistemp_raw, skiprows=1)

# melt monthly columns into long format
gistemp_long = gistemp.melt(
    id_vars=["Year"],
    var_name="month",
    value_name="temp_anomaly"
)

gistemp_yearly = gistemp_long.groupby("Year")["temp_anomaly"].mean().reset_index()

gistemp_yearly.to_csv(PROCESSED / "gistemp_global_yearly.csv", index=False)

### CHIRPS Precipitation Dataset

In [ ]:
url = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/netcdf/chirps-v2.0.monthly.nc"
import xarray as xr

ds = xr.open_dataset(url)

# Bacolod approx:
lat = slice(11.5, 10.5)
lon = slice(122.5, 123.5)

local = ds.sel(latitude=lat, longitude=lon)

rain = local.mean(dim=["latitude", "longitude"])

df_rain = rain.to_dataframe().reset_index()
df_rain.to_csv(PROCESSED / "chirps_bacolod_monthly.csv", index=False)

### Mauna Loa CO2 record -- NOAA Global Monitoring Laboratory

In [ ]:
"""
# ... download
# ... process so that cols [year, co2_ppm]

co2_yearly.to_csv(
    PROCESSED / "mauna_loa_co2_yearly.csv",
    index=False,
)
"""

### ENSO (Niño3.4 Index) -- NOAA Physical Sciences Laboratory

In [ ]:
"""
# ... download

enso = pd.read_csv(...)

enso["year"] = ...
enso_yearly = (
    enso.groupby("year")["nino34"]
    .mean()
    .reset_index()
)

enso_yearly.to_csv(
    PROCESSED / "enso_yearly.csv",
    index=False,
)
"""

### Aerosol Optical Depth (AOD, proxy for volcanic activity) -- NASA GISS
Should appear as large spikes:
- 1963  Agung
- 1982  El Chichón
- 1991  Pinatubo

In [ ]:
# download code here ...

### Total Solar Irradiance (TSI)
Source?

In [ ]:
# download code ...

### Orbital forcing (Milankovitch cycles)
-> Create visual here

In [ ]:
"""
Create visual and save figures/orbital_timescales.png

Then in day1.ipynb we can:

from IPython.display import Image
Image("figures/orbital_timescales.png")

And explain:

Orbital cycles:
  ~20 kyr
  ~40 kyr
  ~100 kyr

Current warming:
  ~100 yr
"""